# VeritaPlugin — treinamento BERTimbau em GPU
Notebook preparado para replicar o código das autoras; **ainda não executado em GPU**.
Abra no Google Colab e selecione um ambiente Python com GPU (T4, se disponível).
Execute as células na ordem. Não é necessária chave da OpenAI.
**Antes de treinar:** aguarde a conclusão das células de ambiente, código/dados e
modelo base. Se alguma falhar, resolva esse erro antes de continuar. Reiniciar a
sessão apaga variáveis; desconectar/remover o runtime também pode apagar os arquivos.
A célula de treino recupera uma única pasta preparada no disco, mas não escolhe
entre múltiplas execuções e não sobrescreve uma tentativa anterior.

O protocolo tem 450 mensagens, seis classes, holdout (361/44/45) e cinco folds.
Preservamos batch 16, até 4 épocas, learning rate 2e-5, peso de regularização 0.01,
512 tokens, seed do Trainer 42 e early stopping com paciência 2.
Os seis ajustes partem de `neuralmind/bert-base-portuguese-cased`.

O notebook instala Python 3.12 e dependências fixadas em um ambiente separado,
mesmo quando o kernel do Colab utiliza outra versão. Não é necessário trocar o kernel.
O bootstrap usa uv 0.13.4; as versões efetivas são salvas nas evidências.
O subprocesso usa MPLBACKEND=Agg e cache local para salvar gráficos sem depender
do backend interativo do Colab.
Referência: https://docs.astral.sh/uv/pip/environments/
A disponibilidade e a duração das GPUs gratuitas não são garantidas:
https://research.google.com/colaboratory/faq.html


In [ ]:
import sys, subprocess, pathlib, json, shutil, datetime, hashlib, os
print('Python do notebook:', sys.version)
if shutil.which('nvidia-smi') is None:
    raise RuntimeError('Ative uma GPU em Ambiente de execução > Alterar tipo de ambiente de execução antes de continuar.')
subprocess.run(['nvidia-smi'], check=True)
ROOT = pathlib.Path('/content/veritaplugin_bert')
ROOT.mkdir(exist_ok=True)
BOOTSTRAP = ROOT / 'uv_bootstrap'
# O kernel do Colab pode usar outra versão; o treinamento roda em Python 3.12 separado.
subprocess.run([sys.executable, '-m', 'pip', 'install', '--target', str(BOOTSTRAP),
                '--upgrade', 'uv==0.13.4'], check=True)
uv_env = os.environ.copy()
uv_env['PYTHONPATH'] = str(BOOTSTRAP)
uv_env['UV_PYTHON_INSTALL_DIR'] = str(ROOT / 'python_builds')
uv_env['UV_CACHE_DIR'] = str(ROOT / 'uv_cache')
VENV = ROOT / 'venv_py312'
PYTHON = str(VENV / 'bin/python')
if not VENV.exists():
    subprocess.run([sys.executable, '-m', 'uv', 'venv', '--python', '3.12',
                    '--managed-python', '--seed', str(VENV)], env=uv_env, check=True)
subprocess.run([PYTHON, '-c',
    'import sys; assert sys.version_info[:2] == (3,12); print("Python do treinamento:", sys.version)'], check=True)
(ROOT/'bootstrap_versoes.txt').write_text(
    'kernel: ' + sys.version + chr(10) +
    subprocess.check_output([PYTHON, '--version'], text=True) +
    subprocess.check_output([sys.executable, '-m', 'uv', '--version'], env=uv_env, text=True))
subprocess.run([PYTHON, '-m', 'pip', 'install', 'torch==2.4.0',
                '--index-url', 'https://download.pytorch.org/whl/cu121'], check=True)
subprocess.run([PYTHON, '-m', 'pip', 'install',
    'transformers==4.44.2', 'datasets==2.21.0', 'accelerate==0.34.2',
    'numpy==1.26.4', 'pandas==2.2.3', 'scikit-learn==1.5.2',
    'matplotlib==3.9.2', 'pyarrow==17.0.0', 'huggingface-hub==0.24.7'], check=True)
subprocess.run([PYTHON, '-c',
    'import torch; assert torch.cuda.is_available(); print(torch.__version__, torch.cuda.get_device_name(0))'], check=True)


## Código e dados fixados
Usamos o commit auditado, sem copiar os resultados publicados para a pasta de execução.
Cada execução recebe uma pasta nova. Os arquivos originais de divisão são preservados.


In [ ]:
REPO = ROOT / 'autoras'
COMMIT = '0487415058c0cc087dfba4f151652cca82e7d3e2'
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/brunanoroes/treinamento-BERTimbau',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--detach',COMMIT],check=True)
assert subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip() == COMMIT
RUN = ROOT / ('execucao_' + datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
RUN.mkdir()
shutil.copytree(REPO / 'dados_bert', RUN / 'dados_bert')
shutil.copy2(REPO / 'treinar_bert.py', RUN / 'treinar_bert_original.py')
manifest = {str(p.relative_to(RUN)): hashlib.sha256(p.read_bytes()).hexdigest()
            for p in RUN.rglob('*') if p.is_file()}
(RUN/'entradas_sha256.json').write_text(json.dumps(manifest,indent=2))
(RUN/'commit.txt').write_text(COMMIT)
(RUN/'requirements-lock.txt').write_text(subprocess.check_output([PYTHON,'-m','pip','freeze'],text=True))
shutil.copy2(ROOT/'bootstrap_versoes.txt', RUN/'bootstrap_versoes.txt')
(RUN/'gpu.txt').write_text(subprocess.check_output(['nvidia-smi'],text=True))
print('Execução:', RUN)


## Fixar a revisão do modelo base
A única adaptação no script de treinamento troca o identificador remoto pelo caminho
local de uma revisão resolvida e registrada. O diff será salvo. A revisão histórica
exata utilizada pelas autoras não está estabelecida, portanto isto não garante
reprodução bit a bit. O classificador publicado não é usado como ponto de partida.


In [ ]:
preparacao = r"""
import pathlib, json, difflib
from huggingface_hub import HfApi, snapshot_download
root = pathlib.Path.cwd()
model = 'neuralmind/bert-base-portuguese-cased'
revision = HfApi().model_info(model).sha
snapshot = snapshot_download(model, revision=revision,
    allow_patterns=['config.json','tokenizer*','vocab.txt','special_tokens_map.json',
                    '*.safetensors','pytorch_model.bin'])
(root/'modelo_base.json').write_text(json.dumps({'repo_id':model,'revision':revision,'snapshot':snapshot},indent=2))
original = (root/'treinar_bert_original.py').read_text()
needle = 'MODEL_NAME  = "neuralmind/bert-base-portuguese-cased"'
assert original.count(needle) == 1
adaptado = original.replace(needle, 'MODEL_NAME  = ' + repr(snapshot))
(root/'treinar_bert.py').write_text(adaptado)
(root/'adaptacao.patch').write_text(''.join(difflib.unified_diff(
    original.splitlines(True),adaptado.splitlines(True),fromfile='original',tofile='execucao')))
"""
(RUN/'preparar_modelo.py').write_text(preparacao)
subprocess.run([PYTHON,'preparar_modelo.py'],cwd=RUN,check=True)


## Executar os seis treinamentos
Esta célula pode levar horas. A duração real será registrada; não é uma previsão.
Em caso de falta de memória, guarde o log. Não reduza batch ou tokens silenciosamente:
essa mudança exigiria registrar um protocolo diferente. Uma GPU com mais memória
pode ser necessária. O script original remove checkpoints intermediários e salva
apenas o modelo final do holdout; não oferece retomada automática.

**Limitações originais preservadas:** no k-fold, o conjunto de teste também orienta
a seleção de época; há duplicatas em alguns folds. O seed do Trainer não é aplicado
explicitamente antes da primeira criação da cabeça de classificação. Esses fatores
precisam acompanhar a interpretação das métricas; não são corrigidos nesta replicação.


In [ ]:
import time, os, sys, subprocess, pathlib, json, datetime
ROOT = pathlib.Path('/content/veritaplugin_bert')
PYTHON = str(ROOT / 'venv_py312/bin/python')
if 'RUN' not in globals():
    candidatas = sorted(p for p in ROOT.glob('execucao_*')
                        if p.is_dir() and (p/'treinar_bert.py').is_file()
                        and (p/'modelo_base.json').is_file())
    if len(candidatas) != 1:
        raise RuntimeError(
            'Preparação ausente ou ambígua. Execute primeiro as células de ambiente, '
            'Código e dados fixados e Fixar a revisão do modelo base, sem erros. '
            'Se houver várias execuções, defina RUN = pathlib.Path(caminho_da_execucao) '
            'explicitamente. Pastas encontradas: ' + str([str(p) for p in candidatas]))
    RUN = candidatas[0]
    print('Pasta recuperada após perda das variáveis:', RUN)
RUN = pathlib.Path(RUN)
necessarios = [pathlib.Path(PYTHON), RUN/'treinar_bert.py', RUN/'dados_bert', RUN/'modelo_base.json']
faltantes = [str(p) for p in necessarios if not p.exists()]
if faltantes:
    raise RuntimeError('Preparação incompleta. Execute as três células anteriores. Faltam: ' + str(faltantes))
modelo_base = json.loads((RUN/'modelo_base.json').read_text())
if not pathlib.Path(modelo_base['snapshot']).is_dir():
    raise RuntimeError('O modelo base não está mais no disco. Execute novamente a célula Fixar a revisão do modelo base.')
if (RUN/'status.json').exists() or (RUN/'treinamento.log').exists() or (RUN/'resultados_bert').exists():
    raise RuntimeError('Esta pasta já contém uma tentativa de treinamento. Preserve os arquivos; '
                       'para começar de novo, execute Código e dados fixados e depois Fixar a revisão do modelo base, '
                       'criando uma pasta nova. O script original não retoma checkpoints automaticamente.')
started = time.time()
status = {'started_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),'status':'running'}
(RUN/'status.json').write_text(json.dumps(status,indent=2))
env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
# O kernel Colab pode exportar o backend interativo para subprocessos isolados.
env['MPLBACKEND'] = 'Agg'
env['MPLCONFIGDIR'] = str(RUN / '.matplotlib')
(RUN / '.matplotlib').mkdir(exist_ok=True)
(RUN/'ambiente_graficos.json').write_text(json.dumps({
    'MPLBACKEND':env['MPLBACKEND'], 'MPLCONFIGDIR':env['MPLCONFIGDIR']},indent=2))
# Verificar importação antes de abrir o log de treinamento.
try:
    subprocess.run([PYTHON, '-c', 'import matplotlib.pyplot as plt; import matplotlib; '
                    'assert matplotlib.get_backend().lower() == "agg"; print("Matplotlib Agg OK")'],
                   cwd=RUN, env=env, check=True)
except BaseException as exc:
    status.update(status='failed_preflight',error_type=type(exc).__name__)
    (RUN/'status.json').write_text(json.dumps(status,indent=2))
    raise
try:
    with (RUN/'treinamento.log').open('w') as log:
        process = subprocess.Popen([PYTHON,'treinar_bert.py'],cwd=RUN,env=env,
                                   stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        try:
            for line in process.stdout:
                log.write(line); log.flush(); print(line,end='')
            returncode = process.wait()
        except BaseException:
            process.terminate()
            try: process.wait(timeout=20)
            except subprocess.TimeoutExpired: process.kill(); process.wait()
            raise
    status.update(status='complete' if returncode == 0 else 'failed',returncode=returncode)
    if returncode != 0:
        raise RuntimeError('Treinamento falhou. Consulte treinamento.log antes de tentar novamente.')
except BaseException as exc:
    status.update(status='failed_or_interrupted',error_type=type(exc).__name__)
    raise
finally:
    status['duration_seconds'] = time.time()-started
    (RUN/'status.json').write_text(json.dumps(status,indent=2))


## Conferir e baixar as evidências
Execute esta célula mesmo após uma falha para preservar o log. O ZIP inclui o modelo
final se ele foi gerado, métricas, gráficos, dados, versões e estado da execução.
Baixe antes que a sessão temporária do Colab seja encerrada. Uma falha ou interrupção
não conta como treinamento concluído. Envie o ZIP para continuarmos a análise.


In [ ]:
print((RUN/'status.json').read_text() if (RUN/'status.json').exists() else 'Treinamento não iniciado')
for filename in ['holdout_metricas.json','kfold_resultado_final.json']:
    path=RUN/'resultados_bert'/filename
    if path.exists(): print(filename, path.read_text())
    else: print(filename, 'NÃO GERADO')
zip_path=shutil.make_archive(str(RUN), 'zip', root_dir=RUN)
from google.colab import files
files.download(zip_path)
